In [ ]:
from custom_openai_client import OpenAIClient
from datetime import datetime
from pathlib import Path
import base64
import os
import json
import pandas as pd
import ast
from tqdm import tqdm

API_KEY = os.environ.get("API_KEY", "ENTER_YOUR_API_KEY")
BASE_URL = os.environ.get("BASE_URL", "https://api.openai.com/v1")
MODEL = os.environ.get("MODEL", "gpt-4o")
TEMPERATURE = float(os.environ.get("TEMPERATURE", 0.7))
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", 8196))

api_config = {
    "api_key": API_KEY,
    "base_url": BASE_URL,
    "model": MODEL,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS
}

openai_client = OpenAIClient(
        api_config=api_config,
        max_retries=3,
        timeout=30,
        rate_limit_rpm=120
    )

In [ ]:
current_date = datetime.now()
formatted_date = current_date.strftime('%Y%m%d')

img_path = Path('/path/to/image/folder')
csv_file_path = Path('/path/to/csv/file')
save_path = Path(f'/path/to/save/folder/conversations_{formatted_date}.json')

In [ ]:
def load_images_to_dict(folder_path):
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp']
    images_dict = {}
    
    folder = Path(folder_path)
    for file_path in folder.iterdir():
        if file_path.is_file() and any(file_path.suffix.lower() == ext for ext in image_extensions):
            try:
                images_dict[file_path.stem] = get_image_base64(file_path)
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
    
    return images_dict

def get_image_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

images_dict = load_images_to_dict(img_path)

In [ ]:
class ConversationSaver:
    def __init__(self, file_path):
        self.file_path = file_path
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                self.all_conversations = json.load(f)
                print(f"Loaded {len(self.all_conversations)} existing conversations from {file_path}")
        except (FileNotFoundError, json.JSONDecodeError):
            self.all_conversations = []
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write('[]')
            print(f"Initialized new conversation file at {file_path}")
    
    def save(self, conversation_record):
        self.all_conversations.append(conversation_record)
        
        temp_file = f"{self.file_path}.temp"
        with open(temp_file, 'w', encoding='utf-8') as f:
            json.dump(self.all_conversations, f, ensure_ascii=False, indent=4)
        
        os.replace(temp_file, self.file_path)
        print(f"Saved record with num: {conversation_record['num']}. Total records: {len(self.all_conversations)}")
    
    def get_all(self):
        return self.all_conversations
            
    def get_processed_ids(self):
        return [conv["num"] for conv in self.all_conversations]
        
    def count(self):
        return len(self.all_conversations)

In [ ]:
def question_generator(csv_file_path, images_dict, save_path):
    saver = ConversationSaver(save_path)
    processed_ids = set(saver.get_processed_ids())
    TEST_MODE = False 

    df = pd.read_csv(csv_file_path)
    if TEST_MODE:
        df = df.head(3)
       
    for index, row in tqdm(df.iterrows(), desc="Generating descriptions", unit="item", total=len(df)):
        num = int(row["num"])
        image_name = str(row["num"])
        value_data = ast.literal_eval(row["value"])
        
        if num in processed_ids:
            print(f"Skipping already processed item: {num}")
            continue
        
        message = [
            {"role": "system",
            "content": 'You are a time series expert. Your task is to describe the time series data and the image plotted from this data using a paragraph.'},
            {"role": "user",
            "content": f"The time series data with detailed values as follows: {value_data}"
            },
            {"role": "user",
            "content": [
                {"type": "text", "text": "After plotting this time series data as an image, with index as x-axis and value as y-axis, the base64 encoded image is as follows:"},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images_dict.get(image_name)}"}}
            ]
            }]
        
        
        message.extend([
            {"role": "user",
            "content": f"""Additionally, you need to follow these rules:
            1. Use concise language to describe this time series data overall, accurately summarizing its characteristics and features.
            2. Do not include too many precise numerical information, use relatively vague descriptive statements.
            3. Do not create fabricated data backgrounds, for example: assume this is financial data.
            4. Do not fabricate phenomena to continue describing, do not describe appearances unrelated to this time series data.
            5. Do not fabricate time, if you need to indicate intervals, use horizontal coordinate indices.
            6. Your output must be in English, descriptions should conform to time series expression standards, control the word count for each time series data description to 10-20 English words.
            7. Strictly output content in the following format:
            Start with <caption(i)> tag, write the description content for the first data, end with </caption(i)>.
            Please ensure strict adherence to this format, do not add any additional explanations or content.
            Format example:
            <caption1> This segment of time-series data ...
            </caption1><caption2>...</caption2><caption3>...
            </caption3><caption4>...</caption4>
            """
            }
            ])
        
        response_raw = openai_client.generate_response(messages=message)
        response = response_raw.choices[0].message.content
        
        conversation_record = {
            "num": num,
            "message": message,
            "response": response
        }
        
        saver.save(conversation_record)
    
    return saver.get_all()

In [ ]:
all_conversations = question_generator(csv_file_path, images_dict, save_path)

In [ ]:
import json
import pandas as pd
import re

def extract_caption_by_index(response, index):
    pattern = f'<caption{index}>(.*?)</caption{index}>'
    match = re.search(pattern, response, re.DOTALL)
    if match:
        caption_content = match.group(1).strip()
        return caption_content
    return None

def update_option_content(original_option, new_contents):
    if not original_option or not isinstance(original_option, str):
        option_text = ""
        for key, content in new_contents.items():
            if content:
                option_text += f"{key}) {content} "
        return option_text.strip()
    
    updated_option = original_option
    for key, content in new_contents.items():
        if content:
            pattern = f'{key}\\)\\s*[^\\n]*?(?=\\s*[A-D]\\)|$)'
            replacement = f'{key}) {content}'
            updated_option = re.sub(pattern, replacement, updated_option)
    
    return updated_option

def process_json_to_csv(json_file_path, csv_file_path, output_csv_path):
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
    except Exception as e:
        print(f"Failed to read JSON file: {e}")
        return
    
    try:
        df = pd.read_csv(csv_file_path)
    except Exception as e:
        print(f"Failed to read CSV file: {e}")
        return
    
    required_columns = ['num', 'question', 'option']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"CSV file missing required columns: {missing_columns}")
        return
    
    updated_count = 0
    for item in json_data:
        if 'num' not in item or 'response' not in item:
            print(f"Skipping item missing required fields: {item}")
            continue
            
        num = item['num']
        response = item['response']
        
        mask = df['num'] == num
        if not mask.any():
            print(f"Row with num={num} not found in CSV")
            continue
        
        row_index = df[mask].index[0]
        
        question_text = 'Given the provided time series numerical or graphical data, select the description from the four options that best corresponds to the data.'
        df.loc[row_index, 'question'] = question_text
        
        caption_contents = {}
        options_map = {1: 'A', 2: 'B', 3: 'C', 4: 'D'}
        
        for i in range(1, 5):
            caption_content = extract_caption_by_index(response, i)
            if caption_content:
                caption_contents[options_map[i]] = caption_content
            else:
                print(f"Caption{i} content not found, num: {num}")
        
        if caption_contents:
            original_option = df.loc[row_index, 'option']
            updated_option = update_option_content(original_option, caption_contents)
            df.loc[row_index, 'option'] = updated_option
        
        updated_count += 1
        print(f"Updated question and option fields for num={num}")
    
    try:
        df.to_csv(output_csv_path, index=False, encoding='utf-8')
        print(f"Successfully updated {updated_count} rows of data, output file: {output_csv_path}")
    except Exception as e:
        print(f"Failed to save CSV file: {e}")

if __name__ == "__main__":
    json_file = "/path/to/json/caption/conversations_20250811.json"
    input_csv = "/path/to/input/original_info.csv"
    output_csv = "/path/to/output/tscption_qa_120.csv"
    
    process_json_to_csv(json_file, input_csv, output_csv)

#### Shuffle

In [ ]:
import pandas as pd
import ast

df = pd.read_csv("/nas/datasets/AAA_TS_Benchmark/MutiTS/caption/tscption_qa_120.csv")

df = df.drop('option_index', axis=1)

df['value'] = df['value'].apply(lambda x: str([ast.literal_eval(x)[0]]) if pd.notna(x) else x)

df.to_csv("/path/to/tscption_qa_120(cleaned).csv", index=False)

print("Processing completed!")

In [ ]:
import pandas as pd
import random
import re

def balance_answers_and_values_random(csv_file_path, output_file_path=None):
    df = pd.read_csv(csv_file_path)
    
    options = ['A', 'B', 'C', 'D']
    num_rows = len(df)
    
    base_count = num_rows // 4
    remainder = num_rows % 4
    
    new_answers = []
    for i, option in enumerate(options):
        count = base_count + (1 if i < remainder else 0)
        new_answers.extend([option] * count)
    
    random.shuffle(new_answers)
    
    def parse_options(option_string):
        pattern = r'([ABCD]\).*?)(?=[ABCD]\)|$)'
        matches = re.findall(pattern, option_string, re.DOTALL)
        
        options_dict = {}
        for match in matches:
            match = match.strip()
            if match.startswith('A)'):
                options_dict['A'] = match
            elif match.startswith('B)'):
                options_dict['B'] = match
            elif match.startswith('C)'):
                options_dict['C'] = match
            elif match.startswith('D)'):
                options_dict['D'] = match
        
        return options_dict
    
    def reconstruct_options(options_dict):
        return '\n'.join([options_dict[key] for key in ['A', 'B', 'C', 'D']])
    
    for i, row in df.iterrows():
        new_answer = new_answers[i]
        original_answer = row['answer']
        
        if original_answer != new_answer:
            option_string = row['option']
            options_dict = parse_options(option_string)
            
            original_content = options_dict[original_answer]
            new_content = options_dict[new_answer]
            
            original_text = original_content[2:].strip()
            new_text = new_content[2:].strip()
            
            options_dict[original_answer] = f"{original_answer}) {new_text}"
            options_dict[new_answer] = f"{new_answer}) {original_text}"
            
            new_option_string = reconstruct_options(options_dict)
            
            df.at[i, 'option'] = new_option_string
        
        df.at[i, 'answer'] = new_answer
    
    if output_file_path is None:
        output_file_path = csv_file_path
    
    df.to_csv(output_file_path, index=False)
    
    print("Answer distribution statistics:", df['answer'].value_counts().sort_index())
    print(f"Results saved to: {output_file_path}")

balance_answers_and_values_random("/path/to/tscption_qa_120(cleaned).csv", "/path/to/ts2caption_qa_120_new.csv")

In [ ]:
import pandas as pd
import ast
import numpy as np

def process_value_column(value_str):
    try:
        if pd.isna(value_str) or value_str == '':
            return value_str
        
        if isinstance(value_str, str):
            value_list = ast.literal_eval(value_str)
        else:
            value_list = value_str
        
        def round_numbers(item):
            if isinstance(item, list):
                return [round_numbers(x) for x in item]
            elif isinstance(item, (int, float)):
                return round(float(item), 2)
            else:
                return item
        
        processed_list = round_numbers(value_list)
        return str(processed_list)
        
    except (ValueError, SyntaxError) as e:
        print(f"Error processing value column: {e}, original value: {value_str}")
        return value_str

def process_csv_file(input_file, output_file):
    print("Reading CSV file...")
    df = pd.read_csv(input_file)
    
    print(f"Original data shape: {df.shape}")
    print(f"Original column names: {list(df.columns)}")
    
    if 'value' in df.columns:
        print("Processing value column, rounding numbers to 2 decimal places...")
        df['value'] = df['value'].apply(process_value_column)
        print("Value column processing completed")
    else:
        print("Warning: value column not found")
    
    if 'domain' in df.columns:
        print("Processing domain column, filling with 'synthetic'...")
        df['domain'] = 'synthetic'
        print("Domain column processing completed")
    else:
        print("Warning: domain column not found")
    
    if 'freq' in df.columns:
        print("Processing freq column, filling with 'unknown'...")
        df['freq'] = 'unknown'
        print("Freq column processing completed")
    else:
        print("Warning: freq column not found")
    
    columns_to_drop = [
        'id', 'ts_id', 'target_id', 'start_index', 'end_index', 
        'source', 'features', 'subsequence_index', 'description'
    ]
    
    existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    missing_columns = [col for col in columns_to_drop if col not in df.columns]
    
    if missing_columns:
        print(f"Note: the following columns do not exist in the data: {missing_columns}")
    
    if existing_columns_to_drop:
        print(f"Dropping columns: {existing_columns_to_drop}")
        df = df.drop(columns=existing_columns_to_drop)
    
    print(f"Processed data shape: {df.shape}")
    print(f"Retained column names: {list(df.columns)}")
    
    print(f"Saving to {output_file}...")
    df.to_csv(output_file, index=False)
    print("File saved successfully!")
    
    return df

def show_column_statistics(df):
    print("\n=== Processing Results Statistics ===")
    
    if 'domain' in df.columns:
        domain_counts = df['domain'].value_counts()
        print(f"\nDomain column statistics:")
        print(domain_counts)
    
    if 'freq' in df.columns:
        freq_counts = df['freq'].value_counts()
        print(f"\nFreq column statistics:")
        print(freq_counts)
    
    if 'value' in df.columns:
        print(f"\nFirst 3 examples of value column:")
        for i, value in enumerate(df['value'].head(3)):
            print(f"Row {i+1}: {value}")

if __name__ == "__main__":
    input_file = "/path/to/merged_all_800.csv"
    output_file = "/path/to/merged_all_800_processed_data.csv"
    
    try:
        processed_df = process_csv_file(input_file, output_file)
        
        print("\nFirst 5 rows of processed data preview:")
        print(processed_df.head())
        
        show_column_statistics(processed_df)
        
        print(f"\nFinal dataset information:")
        print(f"Number of rows: {len(processed_df)}")
        print(f"Number of columns: {len(processed_df.columns)}")
        print(f"Retained columns: {list(processed_df.columns)}")
                
    except FileNotFoundError:
        print(f"Error: Input file {input_file} not found")
        print("Please ensure the file path is correct")
    except Exception as e:
        print(f"Error occurred during processing: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
import pandas as pd
import ast

def calculate_value_length(value_str):
    try:
        value = ast.literal_eval(value_str)
        
        if isinstance(value, list) and len(value) > 0 and isinstance(value[0], list):
            return len(value[0])
        elif isinstance(value, list):
            return len(value)
        else:
            return 0
    except:
        return 0

def process_csv(input_file, output_file=None):
    df = pd.read_csv(input_file)
    
    df['length'] = df['value'].apply(calculate_value_length)
    
    if output_file is None:
        output_file = input_file.replace('.csv', '_with_length.csv')
    
    df.to_csv(output_file, index=False)
    print(f"Processing completed, results saved to: {output_file}")
    
    return df

if __name__ == "__main__":
    input_file = "/path/to/tsqa_1084(with description).csv"
    
    df = process_csv(input_file)
    
    print(df[['value', 'length']].head())